In [ ]:
# =========================
# 0. Install / Imports
# =========================
!pip install joblib scikit-learn pandas -q

import pandas as pd
import joblib
import hashlib
from datetime import datetime

In [ ]:
# =========================
# 1. Load Models
# =========================
demand_model = joblib.load("demand_model.joblib")
failure_model = joblib.load("delivery_failure_model.joblib")
maintenance_model = joblib.load("maintenance_model.joblib")
FEATURES = joblib.load("features.joblib")

print("✅ Models loaded.")

In [ ]:
def build_input_df(
    distance_km,
    num_stops,
    hub_congestion,
    traffic_level,
    weather_rain,
    vehicle_mileage_km,
    days_since_service,
    driver_experience,
    current_load,
):
    input_df = pd.DataFrame([{
        "distance_km": distance_km,
        "num_stops": num_stops,
        "hub_congestion": hub_congestion,
        "traffic_level": traffic_level,
        "weather_rain": weather_rain,
        "vehicle_mileage_km": vehicle_mileage_km,
        "days_since_service": days_since_service,
        "driver_experience": driver_experience,
        "current_load": current_load,
    }])

    return input_df[FEATURES]

SUPPORT_KB is RAG-lite.
RAG-lite is not full RAG yet.

Future upgrade for RAG-lite structured knowledge base to full RAG:
```
docs/
  delivery_sop.md
  customer_support_faq.md
  maintenance_policy.md
  claims_escalation_guide.md
```


In [ ]:
# This section simulates a simplified multi-agent AI architecture.
# Each agent is responsible for a specific operational task,
# and the Control Tower Agent coordinates decisions across agents.

# =========================
# 2. Agent Communication + Logging
# =========================
shared_state = {}
decision_logs = []

# =========================
# 2B. RAG-lite Knowledge Base
# =========================

# RAG-lite: rule-based retrieval from structured knowledge base
# Future upgrade: vector database + semantic retrieval

SUPPORT_KB = {
    "high_delivery_failure": {
        "question": "What should we do if delivery failure risk is high?",
        "answer": "Confirm customer availability, check address accuracy, choose a safer delivery time slot, and notify the customer before dispatch."
    },
    "high_maintenance_risk": {
        "question": "What should we do if maintenance risk is high?",
        "answer": "Avoid assigning the vehicle to long routes. Send it for inspection or allocate a backup vehicle."
    },
    "high_demand": {
        "question": "What should we do if predicted parcel volume is high?",
        "answer": "Prepare extra fleet capacity, allocate more warehouse manpower, and monitor hub congestion."
    },
    "high_overall_risk": {
        "question": "What should we do if overall risk is high?",
        "answer": "Escalate to supervisor review and prioritize preventive actions before dispatch."
    },
    "normal_operations": {
        "question": "What if all risks are low?",
        "answer": "Proceed with normal operations while continuing routine monitoring."
    }
}

# =========================
# 3. Specialist Agents
# =========================
def demand_agent(input_df):
    demand = demand_model.predict(input_df)[0]

    shared_state["demand_agent"] = {
        "predicted_volume": round(float(demand), 2)
    }

    return demand


def delivery_risk_agent(input_df):
    failure_prob = failure_model.predict_proba(input_df)[0][1]

    shared_state["delivery_risk_agent"] = {
        "failure_probability": round(float(failure_prob), 3)
    }

    return failure_prob


def maintenance_agent(input_df):
    maintenance_prob = maintenance_model.predict_proba(input_df)[0][1]

    shared_state["maintenance_agent"] = {
        "maintenance_probability": round(float(maintenance_prob), 3)
    }

    return maintenance_prob

# =========================
# 3B. RAG-lite Support Agent
# =========================
def support_agent(demand, failure_prob, maintenance_prob, final_risk):
    retrieved_answers = []

    if demand >= 150:
        retrieved_answers.append(SUPPORT_KB["high_demand"]["answer"])

    if failure_prob >= 0.5:
        retrieved_answers.append(SUPPORT_KB["high_delivery_failure"]["answer"])

    if maintenance_prob >= 0.5:
        retrieved_answers.append(SUPPORT_KB["high_maintenance_risk"]["answer"])

    if final_risk >= 0.7:
        retrieved_answers.append(SUPPORT_KB["high_overall_risk"]["answer"])

    if not retrieved_answers:
        retrieved_answers.append(SUPPORT_KB["normal_operations"]["answer"])

    shared_state["support_agent"] = {
        "retrieved_guidance": retrieved_answers
    }

    return retrieved_answers

def explain_agent(input_df):
    explanations = []

    row = input_df.iloc[0]

    if row["traffic_level"] > 0.7:
        explanations.append("High traffic level is a major contributor to delivery delay risk.")

    if row["hub_congestion"] > 0.75:
        explanations.append("Severe hub congestion may significantly slow parcel processing.")

    if row["weather_rain"] == 1:
        explanations.append("Rain conditions moderately increase delivery disruption risk.")

    if row["vehicle_mileage_km"] > 160000:
        explanations.append("High vehicle mileage is a strong indicator of breakdown probability.")

    if row["days_since_service"] > 120:
        explanations.append("Overdue servicing increases mechanical failure risk.")

    if row["current_load"] > 70:
        explanations.append("Heavy load increases both handling time and delivery risk.")

    if not explanations:
        explanations.append("No significant risk factors detected.")

    shared_state["explain_agent"] = {
        "explanations": explanations
    }

    return explanations

# =========================
# 4. Control Tower Agent
# =========================

# Control Tower combines outputs from multiple agents
# to produce coordinated operational recommendations

def control_tower_agent(demand, failure_prob, maintenance_prob):
    recommendations = []

    final_risk = (
        0.4 * failure_prob +
        0.4 * maintenance_prob +
        0.2 * min(demand / 250, 1)
    )

    if demand >= 150:
        recommendations.append("Prepare extra fleet capacity.")

    if failure_prob >= 0.5:
        recommendations.append("High delivery failure risk — confirm customer availability.")

    if maintenance_prob >= 0.5:
        recommendations.append("Vehicle maintenance required before dispatch.")

    if final_risk >= 0.7:
        recommendations.append("Overall risk is high — assign supervisor review.")

    if not recommendations:
        recommendations.append("Proceed with normal operations.")

    shared_state["control_tower_agent"] = {
        "final_risk_score": round(float(final_risk), 3),
        "recommendations": recommendations
    }

    return final_risk, recommendations

# =========================
# 5. Master Orchestrator
# =========================
def run_ninjai_system(
    distance_km,
    num_stops,
    hub_congestion,
    traffic_level,
    weather_rain,
    vehicle_mileage_km,
    days_since_service,
    driver_experience,
    current_load,
):
    shared_state.clear()    # Before each new run, clear shared_state inside run_ninjai_system() so old agent results do not accidentally remain.

    input_df = build_input_df(
        distance_km,
        num_stops,
        hub_congestion,
        traffic_level,
        weather_rain,
        vehicle_mileage_km,
        days_since_service,
        driver_experience,
        current_load,
    )

    demand = demand_agent(input_df)
    failure_prob = delivery_risk_agent(input_df)
    maintenance_prob = maintenance_agent(input_df)

    final_risk, recommendations = control_tower_agent(
        demand,
        failure_prob,
        maintenance_prob
    )

    support_guidance = support_agent(
        demand,
        failure_prob,
        maintenance_prob,
        final_risk
    )

    explanations = explain_agent(input_df)

    result = {
        "predicted_volume": round(float(demand), 2),
        "failure_probability": round(float(failure_prob), 3),
        "maintenance_probability": round(float(maintenance_prob), 3),
        "final_risk_score": round(float(final_risk), 3),
        "recommendations": recommendations,
        "support_guidance": support_guidance,
        "explanations": explanations,
        "shared_state": shared_state.copy()
    }

    decision_logs.append(result)

    return result

In [ ]:
# =========================
# 3. Test With 1 Row
# =========================

result = run_ninjai_system(
    distance_km=15,
    num_stops=6,
    hub_congestion=0.8,
    traffic_level=0.7,
    weather_rain=1,
    vehicle_mileage_km=190000,
    days_since_service=130,
    driver_experience=2,
    current_load=85,
)

result

In [ ]:
# For latest log
# decision_logs[-1]

In [ ]:
# For log count
# len(decision_logs)

In [ ]:
# Clear the logs manually.

decision_logs.clear()

In [ ]:
# =========================
# 4. Placeholder dApp Proof-of-Delivery
# =========================
def create_delivery_proof(parcel_id, driver_id, customer_id, delivery_status):
    proof_data = {
        "parcel_id": parcel_id,
        "driver_id": driver_id,
        "customer_id": customer_id,
        "delivery_status": delivery_status,
        "timestamp_utc": datetime.utcnow().isoformat()
    }

    proof_string = str(proof_data)
    proof_hash = hashlib.sha256(proof_string.encode()).hexdigest()

    return {
        "proof_data": proof_data,
        "proof_hash": proof_hash,
        "blockchain_status": "Simulated only - not yet written to blockchain"
    }

In [ ]:
proof = create_delivery_proof(
    parcel_id="NV001",
    driver_id="D001",
    customer_id="C001",
    delivery_status="Delivered"
)

proof